# Robustness Check: Weighted vs. Unweighted Sample Analysis

## Overview
This notebook performs robustness analyses of survey results using post-stratification weights for gender, age, and geography based on ISTAT 2025 data. The analysis compares unweighted and weighted distributions across key demographic and adoption variables to assess potential sampling bias.

The analysis is carried out on `survey_clean_var.tsv`, the filtered and validated version of the original respondents data (based on quality and eligibility filters), also enriched with stratification weights. 

## Key Metrics
- **delta_pp**: Percentage point difference between weighted and unweighted proportions
- **PASS**: delta_pp ≤ 3pp (minimal bias)
- **REVIEW**: 3pp < delta_pp ≤ 5pp (moderate bias)
- **CRITICAL**: delta_pp > 5pp (significant bias)

## Scope
The analyses cover:
- Socioemographics (education and income)
- GenAI Chatbot adoption and usage patterns
- Language and technology usage
- User literacy and intentions
- Non-adoption reasons

Results are compiled into `results/robustness_check_summary.csv`

In [ ]:
import pandas as pd
import os
from pathlib import Path
from scipy import stats
from scipy.stats import norm
import pingouin as pg
from statsmodels.stats.proportion import proportion_confint
from scipy.stats import chi2_contingency
from collections import Counter
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re
sns.set_theme("notebook", style="whitegrid", font_scale=1.3)

In [ ]:
notebook_dir = Path(".").resolve()
project_root = notebook_dir.parent
data_file = project_root / "survey_clean_var.tsv"

In [ ]:
df = pd.read_csv(data_file, sep="\t", encoding='utf-8')
df.head(3)

In [ ]:
# Verify derived variables are present
required_vars = [
    'EducationGroup', 'GeographyGroup', 'GenderGroup', 'IncomeGroup', 'AgeGroup',
    'chatbot_user', 'InfoRetrieval_freq', 'ProblemSolving_freq', 'Learning_freq',
    'ContentCreation_freq', 'Entertainment_freq', 'Creativity_freq', 'weight'
]

missing_vars = [v for v in required_vars if v not in df.columns]
if missing_vars:
    print(f"⚠ WARNING: Missing variables: {missing_vars}")
    print(f"Available columns: {df.columns.tolist()}")
else:
    print(f"✓ All required variables present")
    print(f"  Rows: {len(df)}, Columns: {len(df.columns)}")

In [ ]:
pd.set_option("display.max_rows", None)  
pd.set_option("display.max_columns", None) 

In [ ]:
# check weights for each variables combo
var = ['Q6', 'Q17', 'Q10', 'weight']
combos = df[var].drop_duplicates().sort_values('weight')
print(combos)

In [ ]:
# # Weight check
# print("N responses:", len(df))
# print("Weight sum", df['weight'].sum())

# Variables

In [ ]:
# Extended mapping for all occupations to English
extended_occ_mapping = {
    'In pensione': 'Retired',
    'IT e media': 'IT & Media', 
    'Formazione': 'Education',
    'Ricerca e accademia': 'Research & Academia',
    'Impresa e consulenza aziendale': 'Business & Consulting',
    'Sicurezza e pubblica amministrazione': 'Security & Public Admin',
    'Studente': 'Student',
    'Industria e trasporti': 'Industry & Transport',
    'Sanità': 'Healthcare',
    'Non lavoro al momento': 'Currently Not Working',
    'Finanza': 'Finance',
    'Cultura e spettacolo': 'Culture & Entertainment',
    'Turismo e ristorazione': 'Services & Hospitality',
    'Costruzioni': 'Construction',
    'Agricoltura e ambiente': 'Agriculture & Environment'
}

# Checks definitions 

In [ ]:
def freq_table(df, col, weight_col='weight', categories=None, dropna=True):
    """
    Compute frequencies, unweighted and weighted percentages for a categorical column,
    along with delta and cumulative percentages.
    """
    col_data = df[col]
    w = df[weight_col]

    # Determine categorical responses
    if categories is None:
        categories = sorted(col_data.dropna().unique())
    cat = pd.Categorical(col_data, categories=categories, ordered=False)

    # Unweighted frequencies
    freq_unw = cat.value_counts(dropna=dropna).rename('unw_freq')

    # Weighted frequencies (valid rows only)
    mask_valid = ~cat.isna()
    wfreq = (
        pd.Series(w[mask_valid].values, index=cat[mask_valid])
        .groupby(level=0, observed=True).sum()
        .reindex(categories)
        .fillna(0)
        .rename('w_freq')
    )

    # Percentages
    unw_pct = (freq_unw / (~cat.isna()).sum()).rename('unw_pct')
    # denominator = total weight of valid responses
    den = w[mask_valid].sum()  
    wt_pct = (wfreq / den).rename('w_pct')

    # 
    out = pd.concat([freq_unw, wfreq, unw_pct, wt_pct], axis=1).fillna(0)
    out['delta_pp'] = (out['w_pct'] - out['unw_pct']) * 100
    out['cum_unw'] = out['unw_pct'].cumsum() * 100
    out['cum_w'] = out['w_pct'].cumsum() * 100

    # Convert percentages to %
    out[['unw_pct', 'w_pct']] = out[['unw_pct', 'w_pct']] * 100

    return out.round(2)

In [ ]:
# Flag potential bias based on predefined thresholds
def question_flag(table, delta_pp_thresholds=(3,5)):
    dmax = table['delta_pp'].abs().max()
    if dmax <= delta_pp_thresholds[0]:
        flag = 'PASS'
    elif dmax <= delta_pp_thresholds[1]:
        flag = 'REVIEW'
    else:
        flag = 'CRITICAL'
    #print(f"Question flag: {flag}, delta_pp={dmax:.2f}")
    return flag, dmax

In [ ]:
def batch_dashboard(df, questions, weight_col='weight', categories_map=None):
    rows = []
    # Get categorical responses
    for q in questions:
        cats = None if categories_map is None else categories_map.get(q)
        # Compute the frequency table for this question
        tab = freq_table(df, q, weight_col, categories=cats)
        flag, dmax = question_flag(tab)
        # Check if the top-2 responses match (1) or not (0) between unweighted and weighted percentages
        top2_unw = tab['unw_pct'].sort_values(ascending=False).index[:2].tolist()
        top2_w   = tab['w_pct'].sort_values(ascending=False).index[:2].tolist()
        top2_same = int(top2_unw == top2_w)
        rows.append({'question': q, 'delta_max_pp': float(dmax),
                     'top2_same': top2_same, 'flag': flag})
    # sort by flag and delta
    dash = pd.DataFrame(rows).sort_values(['flag','delta_max_pp'], ascending=[True, False])
    return dash

# Apply robustness checks

## Education

In [ ]:
# Education
questions = ['EducationGroup']
dashboard = batch_dashboard(df, questions)
print(dashboard)

In [ ]:
edu_table = freq_table(df, 'EducationGroup', weight_col='weight')
print(edu_table)

## Chatbot users

In [ ]:
questions = ['chatbot_user']
dashboard = batch_dashboard(df, questions)
print(dashboard)

In [ ]:
chatbot_table = freq_table(df, 'chatbot_user', weight_col='weight')
print(chatbot_table)

## Socioeconomic

In [ ]:
questions = ['IncomeGroup']
dashboard = batch_dashboard(df, questions)
print(dashboard)

In [ ]:
income_table = freq_table(df, 'IncomeGroup', weight_col='weight')
print(income_table)

## LT adoption and replacement

In [ ]:
# Define mappings to calculate proportions relative the N of users of the specific application
lt_mapping = {
    'Q61': 'Q20-MT',             # MT
    'Q62': 'Q34',                # Speech Transcript
    'Q63': 'Q21',                # Vocal Assistants
    'Q64': 'Q33',                # Assisted Writing
    'Q65': 'Q36',                # Text-to-Speech
    'Q66': 'Q43'                 # Web Search replacement calculated among GenAI users
}

In [ ]:
# Create binary usage columns (1 = uses, 0 = doesn't use)
for rep_q, use_q in lt_mapping.items():
    if use_q:
        used_mask = ~df[use_q].str.contains("Non ho mai usato", na=False, case=False)
        df[f'{rep_q}_uses_tech'] = used_mask.astype(int)
        df.loc[~used_mask, f'{rep_q}_uses_tech'] = 0
    else:
        df[f'{rep_q}_uses_tech'] = 1  


# Create binary replacement columns (among users only)
for rep_q, use_q in lt_mapping.items():
    if use_q:
        used_mask = ~df[use_q].str.contains("Non ho mai usato", na=False, case=False)
    else:
        used_mask = df[rep_q].notna()
    
    # Completely replaced (binary: 1 = yes, 0 = no, NaN = non-user)
    df[f'{rep_q}_comp_replaced'] = np.nan
    df.loc[used_mask, f'{rep_q}_comp_replaced'] = (
        df.loc[used_mask, rep_q] == "Completamente sostituito"
    ).astype(int)
    
    # Partially replaced (binary: 1 = yes, 0 = no, NaN = non-user)
    df[f'{rep_q}_parz_replaced'] = np.nan
    df.loc[used_mask, f'{rep_q}_parz_replaced'] = (
        df.loc[used_mask, rep_q] == "Parzialmente sostituito"
    ).astype(int)

usage_cols = [f'{rep_q}_uses_tech' for rep_q in lt_mapping.keys()]
comp_cols = [f'{rep_q}_comp_replaced' for rep_q in lt_mapping.keys()]
parz_cols = [f'{rep_q}_parz_replaced' for rep_q in lt_mapping.keys()]

In [ ]:
usage_dashboard = batch_dashboard(df, usage_cols)
comp_dashboard = batch_dashboard(df, comp_cols)
parz_dashboard = batch_dashboard(df, parz_cols)

In [ ]:
print(usage_dashboard)

In [ ]:
print(comp_dashboard)

In [ ]:
print(parz_dashboard)

In [ ]:
print("\nQ64 Parz Replaced:")
print(freq_table(df, 'Q64_parz_replaced', weight_col='weight'))

## Check reasons for replacement

In [ ]:
questions = ['Q67']
dashboard = batch_dashboard(df, questions)
print(dashboard)

## Intent Frequency

In [ ]:
# Define intent mapping
intent_mapping = {
    'Q51_1': 'InfoRetrieval',
    'Q51_2': 'ProblemSolving',
    'Q51_3': 'Learning',
    'Q51_4': 'ContentCreation',
    'Q51_5': 'Entertainment',
    'Q51_6': 'Creativity'
}

# Get all frequency columns
freq_columns = [f'{new_col}_freq' for new_col in intent_mapping.values()]

intent_dashboard = batch_dashboard(df, freq_columns)
print(intent_dashboard)

In [ ]:
print(freq_table(df, 'Learning_freq', weight_col='weight'))

## Work and personal use by occupation

In [ ]:
# Activity labels
q53_to_label_en = {
    "Q53_1": "Email Writing", "Q53_2": "Creative Writing", "Q53_3": "Academic Writing",
    "Q53_4": "Other Writing", "Q53_5": "Text Analysis", "Q53_6": "Idea Generation",
    "Q53_7": "Explanations", "Q53_8": "Summaries", "Q53_9": "Quiz Creation",
    "Q53_10": "Travel Planning", "Q53_11": "Fact Checking", "Q53_12": "Personal Advice",
    "Q53_13": "Medical Advice", "Q53_14": "Other Advice", "Q53_15": "Emotional Support",
    "Q53_16": "Casual Chat", "Q53_17": "Romantic Chat", "Q53_18": "Philosophical Chat",
    "Q53_19": "Other Chat", "Q53_20": "Programming Help", "Q53_21": "Data Analysis",
    "Q53_22": "Image Generation", "Q53_23": "Audio/Music Gen.", "Q53_25": "Other"
}

In [ ]:
q53_columns = [f'Q53_{i}' for i in range(1, 26) if i != 24]

# Calculate totals for each person
df['total_work_responses'] = 0
df['total_personal_responses'] = 0

for col in q53_columns:
    if col in df.columns:
        df['total_work_responses'] += df[col].fillna('').str.contains('Studio/Lavoro', case=False).astype(int)
        df['total_personal_responses'] += df[col].fillna('').str.contains('Svago/Uso personale', case=False).astype(int)

# For each occupation, create a work/personal column (only for chatbot users)
if 'mapped_job' in df.columns and 'chatbot_user' in df.columns:
    chatbot_mask = df['chatbot_user'] == 'Yes'
    
    for occ_it, occ_en in extended_occ_mapping.items():
        if occ_it not in ['Altro', 'Agricoltura e ambiente', "Costruzioni"]:
            col_name = f'usage_type_{occ_en.replace(" ", "_").replace("&", "and")}'
            
            # Initialize as object type (string) instead of float
            df[col_name] = pd.NA  # Use pd.NA for string columns
            df[col_name] = df[col_name].astype('object')  # Ensure it's object type
            
            # For this occupation's chatbot users, assign Work or Personal based on majority
            occ_mask = chatbot_mask & (df['mapped_job'] == occ_it)
            has_responses = (df['total_work_responses'] + df['total_personal_responses']) > 0
            
            # Assign "Work" if work responses >= personal, else "Personal"
            work_majority = df['total_work_responses'] >= df['total_personal_responses']
            
            df.loc[occ_mask & has_responses & work_majority, col_name] = 'Work'
            df.loc[occ_mask & has_responses & ~work_majority, col_name] = 'Personal'

# Get all occupation usage columns
occupation_usage_cols = [f'usage_type_{occ_en.replace(" ", "_").replace("&", "and")}' 
                         for occ_it, occ_en in extended_occ_mapping.items()
                         if occ_it not in ['Altro', 'Agricoltura e ambiente', "Costruzioni"]]

In [ ]:
# Use your dashboard
usage_dashboard = batch_dashboard(df, occupation_usage_cols)
print(usage_dashboard)

In [ ]:
print(freq_table(df, 'usage_type_Services_and_Hospitality', weight_col='weight'))

## Usage modalities and strategies

In [ ]:
questions = ['strategies']
dashboard = batch_dashboard(df, questions)
print(dashboard)

In [ ]:
questions = ['access', 'interaction_mode']
dashboard = batch_dashboard(df, questions)
print(dashboard)

## Language Used

In [ ]:
# Get all unique languages
all_languages = (df['language_used'].dropna().str.split(',').explode().str.strip().unique())

# Create binary column for each language (1 = uses this language, 0 = doesn't, NaN = no answer)
for lang in all_languages:
    col_name = f'uses_{lang.replace(" ", "_")}'
    
    # Initialize as NaN
    df[col_name] = np.nan
    
    # For people who answered language_used
    has_answer = df['language_used'].notna()
    
    # 1 if this language is in their list, 0 if not
    df.loc[has_answer, col_name] = df.loc[has_answer, 'language_used'].str.contains(
        lang, case=False, na=False, regex=False
    ).astype(int)

# Get all language columns
language_cols = [f'uses_{lang.replace(" ", "_")}' for lang in all_languages]

In [ ]:
# Use your dashboard
language_dashboard = batch_dashboard(df, language_cols)
print(language_dashboard)

In [ ]:
print(freq_table(df, 'uses_Inglese', weight_col='weight'))

## Experience with errors and Bias

In [ ]:
questions = ['personal_experience']
dashboard = batch_dashboard(df, questions)
print(dashboard)

## LT literacy and desiderata

In [ ]:
questions = ['likert_1_1', 'likert_1_3', 'likert_1_4', 'likert_1_5', 'likert_1_6', 'likert_2_1', 'likert_2_2', 'likert_2_3']
dashboard = batch_dashboard(df, questions)
print(dashboard)

## Prior LT education

In [ ]:
questions = ['education_technology']
dashboard = batch_dashboard(df, questions)
print(dashboard)

## Questions for non-users

In [ ]:
questions = ['never_used', 'reason_never_used', 'future_use']
dashboard = batch_dashboard(df, questions)
print(dashboard)

# Export Robustness Summary

In [ ]:
results_dir = project_root / "results"
# Compile all robustness checks into summary
all_checks = []

# Core demographic checks
core_questions = {
    'Demographics': ['EducationGroup'],
    'Adoption': ['chatbot_user'],
    'Socioeconomic': ['IncomeGroup']
}

for category, questions in core_questions.items():
    valid_questions = [q for q in questions if q in df.columns]
    if valid_questions:
        dashboard = batch_dashboard(df, valid_questions)
        dashboard['category'] = category
        all_checks.append(dashboard)

# Technology adoption checks
if all(col in df.columns for col in ['Q61_uses_tech', 'Q62_uses_tech', 'Q63_uses_tech', 'Q64_uses_tech', 'Q65_uses_tech', 'Q66_uses_tech']):
    tech_cols = ['Q61_uses_tech', 'Q62_uses_tech', 'Q63_uses_tech', 'Q64_uses_tech', 'Q65_uses_tech', 'Q66_uses_tech']
    dashboard = batch_dashboard(df, tech_cols)
    dashboard['category'] = 'Technology Usage'
    all_checks.append(dashboard)

# Technology adoption - Complete Replacement
comp_cols_to_check = [f'{rep_q}_comp_replaced' for rep_q in lt_mapping.keys()]
valid_comp_cols = [col for col in comp_cols_to_check if col in df.columns]
if valid_comp_cols:
    dashboard = batch_dashboard(df, valid_comp_cols)
    dashboard['category'] = 'Technology Replacement (Complete)'
    all_checks.append(dashboard)

# Technology adoption - Partial Replacement
parz_cols_to_check = [f'{rep_q}_parz_replaced' for rep_q in lt_mapping.keys()]
valid_parz_cols = [col for col in parz_cols_to_check if col in df.columns]
if valid_parz_cols:
    dashboard = batch_dashboard(df, valid_parz_cols)
    dashboard['category'] = 'Technology Replacement (Partial)'
    all_checks.append(dashboard)

# Reasons for replacement
if 'Q67' in df.columns:
    dashboard = batch_dashboard(df, ['Q67'])
    dashboard['category'] = 'Replacement Reasons'
    all_checks.append(dashboard)

# Intent frequency checks
if all(col in df.columns for col in ['InfoRetrieval_freq', 'ProblemSolving_freq', 'Learning_freq', 'ContentCreation_freq', 'Entertainment_freq', 'Creativity_freq']):
    intent_cols = ['InfoRetrieval_freq', 'ProblemSolving_freq', 'Learning_freq', 'ContentCreation_freq', 'Entertainment_freq', 'Creativity_freq']
    dashboard = batch_dashboard(df, intent_cols)
    dashboard['category'] = 'Usage Intent Frequency'
    all_checks.append(dashboard)

# Occupation-based usage
occupation_usage_cols_to_check = [col for col in occupation_usage_cols if col in df.columns]
if occupation_usage_cols_to_check:
    dashboard = batch_dashboard(df, occupation_usage_cols_to_check)
    dashboard['category'] = 'Occupation Usage Type'
    all_checks.append(dashboard)

# Usage modalities (strategies, access, interaction)
usage_modalities = []
for col in ['strategies', 'access', 'interaction_mode']:
    if col in df.columns:
        usage_modalities.append(col)
if usage_modalities:
    dashboard = batch_dashboard(df, usage_modalities)
    dashboard['category'] = 'Usage Modalities'
    all_checks.append(dashboard)

# Language usage
if language_cols:
    dashboard = batch_dashboard(df, language_cols)
    dashboard['category'] = 'Language Used'
    all_checks.append(dashboard)

# Experience with errors and bias
if 'personal_experience' in df.columns:
    dashboard = batch_dashboard(df, ['personal_experience'])
    dashboard['category'] = 'Experience'
    all_checks.append(dashboard)

# Literacy and desiderata
literacy_questions = ['likert_1_1', 'likert_1_3', 'likert_1_4', 'likert_1_5', 'likert_1_6', 'likert_2_1', 'likert_2_2', 'likert_2_3']
literacy_questions = [q for q in literacy_questions if q in df.columns]
if literacy_questions:
    dashboard = batch_dashboard(df, literacy_questions)
    dashboard['category'] = 'Literacy & Desiderata'
    all_checks.append(dashboard)

# Prior education
if 'education_technology' in df.columns:
    dashboard = batch_dashboard(df, ['education_technology'])
    dashboard['category'] = 'Prior Education'
    all_checks.append(dashboard)

# Non-users
non_user_questions = ['never_used', 'reason_never_used', 'future_use']
non_user_questions = [q for q in non_user_questions if q in df.columns]
if non_user_questions:
    dashboard = batch_dashboard(df, non_user_questions)
    dashboard['category'] = 'Non-Users'
    all_checks.append(dashboard)

# Compile summary
if all_checks:
    summary_df = pd.concat(all_checks, ignore_index=True)
    summary_df = summary_df[['category', 'question', 'delta_max_pp', 'top2_same', 'flag']].sort_values(['flag', 'delta_max_pp'], ascending=[True, False])
    
    # Save to CSV
    output_file = results_dir / "robustness_check_summary.csv"
    summary_df.to_csv(output_file, index=False)
    
    print(f"\nSummary Statistics:")
    print(f"  Total checks: {len(summary_df)}")
    print(f"  PASS: {(summary_df['flag'] == 'PASS').sum()}")
    print(f"  REVIEW: {(summary_df['flag'] == 'REVIEW').sum()}")
    print(f"  CRITICAL: {(summary_df['flag'] == 'CRITICAL').sum()}")
    print(f"\nThresholds: PASS ≤ 3pp, REVIEW ≤ 5pp, CRITICAL > 5pp")
    print(f"\nDetailed Results:")
    print(f"\n{summary_df.to_string(index=False)}")